In [2]:
import pandas as pd
df = pd.read_csv('../data/words10_cleaned.csv')
df.head()

,model,language,1,2,3,4,5,6,7,8,9,10
0,gpt_53_chat,english,pebble,thunder,pillow,forest,mirror,bread,shadow,ocean,silence,ladder
1,gpt_53_chat,english,mountain,spoon,thunder,library,childhood,ocean,mirror,bread,shadow,melody
2,gpt_53_chat,english,mountain,toothbrush,democracy,kitten,thunder,pillow,rainbow,hunger,library,shadow
3,gpt_53_chat,french,montagne,chaise,tempete,parfum,miroir,silence,riviere,enfance,liberte,sourire
4,gpt_53_chat,french,arbre,liberte,cuillere,tempete,parfum,horloge,sable,musique,fenetre,sourire


## Embeddings

In [3]:
from sentence_transformers import SentenceTransformer

c:\Users\jipwu\OneDrive\Bureaublad\Projects\campus_numerique\llm_vs_human\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [4]:
# Calculate embeddings
model = SentenceTransformer('paraphrase-multilingual-MiniLM-L12-v2')

df_embeddings = df.copy()

word_cols = df.columns[2:]

for col in word_cols:
    df_embeddings[col] = df_embeddings[col].apply(lambda w: model.encode(w))

df_embeddings.to_pickle('../data/words10_embeddings_paraphrase-multilingual-MiniLM-L12-v2.pkl')

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 2615.16it/s]
BertModel LOAD REPORT from: sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [5]:
# Calculate embeddings
model = SentenceTransformer('LaBSE')

df_embeddings = df.copy()

word_cols = df.columns[2:]

for col in word_cols:
    df_embeddings[col] = df_embeddings[col].apply(lambda w: model.encode(w))

df_embeddings.to_pickle('../data/words10_embeddings_LaBSE.pkl')

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 2939.09it/s]
BertModel LOAD REPORT from: sentence-transformers/LaBSE
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [6]:
import fasttext.util
fasttext.util.download_model('en', if_exists='ignore')  # English
fasttext.util.download_model('fr', if_exists='ignore')  # French
ft_en = fasttext.load_model('cc.en.300.bin')
ft_fr = fasttext.load_model('cc.fr.300.bin')

In [7]:
import numpy as np

df_embeddings = df.copy()
word_cols = df.columns[2:]

def get_embedding(word, lang):
    
    if lang == " english":
        return ft_en.get_word_vector(word)
    elif lang == " french":
        return ft_fr.get_word_vector(word)
    else:
        return np.nan  # fallback if unknown language

for col in word_cols:
    df_embeddings[col] = [
        get_embedding(word, lang)
        for word, lang in zip(df[col], df["language"])
    ]

df_embeddings.to_pickle('../data/words10_embeddings_fasttext.pkl')

In [9]:
from gensim.models import KeyedVectors

#model_en = KeyedVectors.load_word2vec_format("../models/wiki.en.align.vec")
#model_en.save("../models/wiki.multi.en.kv")
#model_fr = KeyedVectors.load_word2vec_format("../models/wiki.fr.align.vec")
#model_fr.save("../models/wiki.multi.fr.kv")

model_en = KeyedVectors.load("../models/wiki.multi.en.kv", mmap='r')
model_fr = KeyedVectors.load("../models/wiki.multi.fr.kv", mmap='r')

In [10]:
import numpy as np

df_embeddings = df.copy()
word_cols = df.columns[2:]

def get_embedding(word, lang):
    if not isinstance(word, str) or word.strip() == "":
        return np.zeros(300)

    model = model_fr if lang.strip() == "french" else model_en

    try:
        return model[word]
    except KeyError:
        return np.zeros(300)

for col in word_cols:
    df_embeddings[col] = [
        get_embedding(word, lang)
        for word, lang in zip(df[col], df["language"])
    ]

df_embeddings.to_pickle('../data/words10_embeddings_fasttext_aligned.pkl')